# Bootcamp Day 3 — scikit-learn

**Companion to bootcamp deck Day 3.** Real datasets: **Iris** (sklearn built-in) + **Titanic-like**.

Basic vocab: `train_test_split`, `fit`, `predict`, `score`, `accuracy_score`, `classification_report`, `cross_val_score`, `LogisticRegression`, `RandomForestClassifier`.

The mantra: **split → fit → predict → score.**

<a href="https://colab.research.google.com/github/Petkub/MachineLearningLab/blob/main/colab_exercises/bootcamp_day3_sklearn.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets        import load_iris
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model    import LogisticRegression
from sklearn.ensemble        import RandomForestClassifier
from sklearn.metrics         import accuracy_score, classification_report
print("ready")

---
## Problem 1 — load Iris

Iris ships with sklearn. 150 flowers, 4 features, 3 classes.

1. `X` — feature matrix
2. `y` — target labels
3. `X_shape` — shape of X (tuple)
4. `n_classes` — number of unique classes in y

In [ ]:
# TODO
X, y = ...

X_shape = ...
n_classes = ...

print("X shape:", X_shape)
print("n_classes:", n_classes)

In [ ]:
assert X_shape == (150, 4)
assert n_classes == 3
print("Q1 ok")

<details><summary>Hint</summary>

```python
X, y = load_iris(return_X_y=True)
X_shape = X.shape
n_classes = len(np.unique(y))
```

</details>

---
## Problem 2 — split into train/test

1. Split Iris with `test_size=0.2`, `random_state=42`
2. `X_train, X_test, y_train, y_test` — the 4 outputs
3. Verify: test set is 30 rows (20% of 150)

In [ ]:
X, y = load_iris(return_X_y=True)

# TODO
X_train, X_test, y_train, y_test = ...

print("train:", X_train.shape, "test:", X_test.shape)

In [ ]:
assert X_train.shape == (120, 4)
assert X_test.shape == (30, 4)
assert y_train.shape == (120,)
assert y_test.shape == (30,)
print("Q2 ok")

<details><summary>Hint</summary>

```python
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
```

Always set `random_state` for reproducibility.

</details>

---
## Problem 3 — train LogReg + score

1. Create `LogisticRegression(max_iter=1000)`
2. Fit on train
3. Predict on test
4. Compute `accuracy` with `accuracy_score`

In [ ]:
X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# TODO
model    = ...
...                              # fit
y_pred   = ...
accuracy = ...

print("accuracy:", accuracy)

In [ ]:
assert accuracy > 0.9
print("Q3 ok — accuracy:", round(accuracy, 3))

<details><summary>Hint</summary>

```python
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
```

</details>

---
## Problem 4 — RandomForest + compare

Now train RandomForest. Does it beat LogReg on Iris?

1. `rf` — `RandomForestClassifier(random_state=42)`
2. `rf_acc` — test accuracy
3. Compare to `logreg_acc` (from Q3 logic)

In [ ]:
X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# TODO
rf = ...
...                       # fit
rf_acc = ...

logreg = LogisticRegression(max_iter=1000).fit(X_train, y_train)
logreg_acc = logreg.score(X_test, y_test)

print("LogReg accuracy:", round(logreg_acc, 3))
print("RF     accuracy:", round(rf_acc, 3))

In [ ]:
assert rf_acc > 0.9
print("Q4 ok")

<details><summary>Hint</summary>

```python
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)
rf_acc = rf.score(X_test, y_test)
```

`.score()` does predict + accuracy in one call.

</details>

---
## Problem 5 — read the classification report

Train RandomForest on Iris. Print `classification_report`.

1. `report` — the report STRING
2. Eye-check: per-class precision, recall, F1 all > 0.85?

In [ ]:
X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
rf = RandomForestClassifier(random_state=42).fit(X_train, y_train)

# TODO
y_pred = ...
report = ...

print(report)

In [ ]:
assert "precision" in report
assert "recall" in report
print("Q5 ok — read the report above")

<details><summary>Hint</summary>

```python
y_pred = rf.predict(X_test)
report = classification_report(y_test, y_pred)
```

The report shows precision, recall, F1, support per class + overall accuracy.

</details>

---
## Problem 6 — cross-validation

One train/test split = noisy. 5-fold CV averages 5 splits.

1. `scores` — 5-fold CV scores using `cross_val_score(rf, X, y, cv=5)`
2. `mean_score` — average
3. `std_score` — standard deviation

In [ ]:
X, y = load_iris(return_X_y=True)
rf = RandomForestClassifier(random_state=42)

# TODO
scores     = ...
mean_score = ...
std_score  = ...

print("scores    :", scores.round(3))
print("mean ± std:", round(mean_score, 3), "±", round(std_score, 3))

In [ ]:
assert scores.shape == (5,)
assert mean_score > 0.9
print("Q6 ok")

<details><summary>Hint</summary>

```python
scores = cross_val_score(rf, X, y, cv=5)
mean_score = scores.mean()
std_score  = scores.std()
```

Defendable number = mean ± std, not a single split.

</details>

---
## Problem 7 — Titanic-like full pipeline

Synthetic Titanic data. Build the pipeline:
1. Clean: fill `Age` NaN with median, drop `Name`
2. Encode: `pd.get_dummies` on `Sex`
3. Split: 80/20, `stratify=y`, `random_state=42`
4. Fit RandomForest
5. Print accuracy + classification_report

In [ ]:
rng = np.random.default_rng(0)
n = 300
df = pd.DataFrame({
    "Name":     [f"P{i}" for i in range(n)],
    "Age":      np.where(rng.random(n) > 0.15, rng.integers(1, 80, size=n), np.nan),
    "Fare":     rng.uniform(5, 100, size=n),
    "Sex":      rng.choice(["male", "female"], size=n),
    "Survived": rng.integers(0, 2, size=n),
})
print(df.head())
print("missing Age:", df["Age"].isna().sum())

# TODO
df["Age"] = ...
df = df.drop(columns=...)
df = pd.get_dummies(df, columns=...)

X = df.drop(columns=["Survived"])
y = df["Survived"]

X_train, X_test, y_train, y_test = ...

model = ...
...                         # fit
y_pred = ...

acc = ...
print("accuracy:", round(acc, 3))
print(classification_report(y_test, y_pred))

In [ ]:
assert 0 <= acc <= 1
assert df["Age"].isna().sum() == 0, "didn't fill Age NaN"
assert "Sex" not in df.columns, "didn't encode Sex"
print("Q7 ok")

<details><summary>Hint</summary>

```python
df["Age"] = df["Age"].fillna(df["Age"].median())
df = df.drop(columns=["Name"])
df = pd.get_dummies(df, columns=["Sex"])

X = df.drop(columns=["Survived"])
y = df["Survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
```

</details>

---
## Problem 8 — find the bugs

The code below "trains a model and gets 100% accuracy." Find and fix **three bugs**.

In [ ]:
X, y = load_iris(return_X_y=True)

# BROKEN — fix three things
model = RandomForestClassifier()                  # bug 1
model.fit(X, y)                                   # bug 2
y_pred = model.predict(X)                         # bug 3
acc = accuracy_score(y, y_pred)
print("accuracy:", acc)                           # 1.0 ✨ — too good to be true

# Save your fixed pipeline:
fixed_acc = ...   # the HONEST accuracy on UNSEEN data

In [ ]:
assert 0.7 < fixed_acc < 1.0, "real test accuracy should be reasonable but not perfect"
print("Q8 ok — honest accuracy:", round(fixed_acc, 3))

<details><summary>Hint</summary>

Bugs:
1. No `random_state` → can't reproduce results.
2. Fitting on ALL data (no split) → model sees everything.
3. Predicting on training data → trivial 100% (memorized).

Fix: split first, fit on train, predict on test.

```python
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestClassifier(random_state=42)
model.fit(X_tr, y_tr)
fixed_acc = accuracy_score(y_te, model.predict(X_te))
```

</details>

---
## Solutions

<details><summary>Show all</summary>

```python
# Q1
X, y = load_iris(return_X_y=True)
X_shape = X.shape
n_classes = len(np.unique(y))

# Q2
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Q3
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

# Q4
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)
rf_acc = rf.score(X_test, y_test)

# Q5
y_pred = rf.predict(X_test)
report = classification_report(y_test, y_pred)

# Q6
scores = cross_val_score(rf, X, y, cv=5)
mean_score = scores.mean()
std_score  = scores.std()

# Q7
df["Age"] = df["Age"].fillna(df["Age"].median())
df = df.drop(columns=["Name"])
df = pd.get_dummies(df, columns=["Sex"])
X = df.drop(columns=["Survived"])
y = df["Survived"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
model = RandomForestClassifier(random_state=42).fit(X_train, y_train)
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)

# Q8 — fixes
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestClassifier(random_state=42)
model.fit(X_tr, y_tr)
fixed_acc = accuracy_score(y_te, model.predict(X_te))
```

</details>